# QE pipeline: 12 комбинаций × 2 датасета

**Ретриверы**: `bm25`, `giga-embeddings`, `RRF(giga + bm25)` (вариант B — расширение применяется только к bm25-ветке).

**QE-методы**: `Query2doc`, `PromptPRF`, `PQR`, `Word2Passage` — все через **DeepSeek API**.

**Датасеты**: `kaengreg/rus-scifact`, `kaengreg/rus-nfcorpus` (полные test-split).

## Алгоритм

Запросы и документы препроцессятся **одинаково**: lowercase + токенизация + лемматизация (pymorphy3) + удаление пунктуации + мягкая фильтрация стоп-слов (без вопросительных и отрицаний). Колонки `processed_text` / `processed_title` из датасетов **не используются**.

Для каждой комбинации `(retriever, qe_method, dataset)`:

1. Берём оригинальный `query.text`.
2. Применяем QE-метод → получаем расширение (генерация на сыром тексте, без предобработки).
3. `expanded = original + ' ' + expansion`.
4. `processed = preprocess(expanded)` — для bm25-ветки.
5. Поиск top-100. Для giga (как самостоятельного ретривера) подаётся `expanded` сырым (с E5-style префиксом). Для giga-ветки в RRF-B подаётся `original` сырым.

Корпус препроцессится **один раз на датасет** тем же `preprocess()` и кэшируется на диск (`cache/corpus_proc_{dataset}.json`).

## Артефакты

* **24 файла расширений** в `expansions/{retriever}_{qe_method}_{dataset}.json` — тройки `{qid, original, expanded, processed}`.
* **24 файла прогонов** в `runs/{retriever}_{qe_method}_{dataset}.json` — top-100 doc_id со скорами на запрос (для будущего реранкера).
* **Сводная таблица метрик** в `results/qe_summary.csv`.

## 0. Установка зависимостей

In [ ]:
!pip install -q datasets 'transformers==4.51.0' 'sentence-transformers==5.1.1' accelerate faiss-gpu-cu12 rank-bm25 pytrec_eval tqdm einops openai pymorphy3 scikit-learn pandas nest_asyncio

In [ ]:
from __future__ import annotations

import os
import re
import gc
import json
import math
import time
import asyncio
from collections import Counter, defaultdict
from pathlib import Path
from typing import Callable

import numpy as np
import torch
import faiss
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm
import pytrec_eval
import pymorphy3
from rank_bm25 import BM25Okapi

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

# --- Директории артефактов -----------------------------------------------
ROOT = Path('.')
EXP_DIR = ROOT / 'expansions'   # 24 файла расширений
RUN_DIR = ROOT / 'runs'         # 24 файла прогонов
RES_DIR = ROOT / 'results'      # сводные метрики
CACHE_DIR = ROOT / 'cache'      # эмбеддинги, BM25-индексы и т.д.
for d in (EXP_DIR, RUN_DIR, RES_DIR, CACHE_DIR):
    d.mkdir(exist_ok=True)

# --- DeepSeek API ---------------------------------------------------------
DEEPSEEK_API_KEY_PLACEHOLDER'DEEPSEEK_API_KEY', '')
DEEPSEEK_MODEL = 'deepseek-chat'
DEEPSEEK_BASE_URL = 'https://api.deepseek.com'
DEEPSEEK_MAX_CONCURRENCY = 32  # параллельных запросов к API

if not DEEPSEEK_API_KEY_PLACEHOLDER    print('WARNING: DEEPSEEK_API_KEY не задан в env — установите перед прогоном QE-методов.')

## 1. Загрузка датасетов

In [ ]:
def load_beir_like(name: str, qrels_split: str = 'test'):
    corpus_ds = load_dataset(f'kaengreg/{name}', 'corpus')['train']
    queries_ds = load_dataset(f'kaengreg/{name}', 'queries')['train']
    qrels_ds = load_dataset(f'kaengreg/{name}-qrels', 'qrels')[qrels_split]

    corpus = {}
    for row in corpus_ds:
        corpus[str(row['_id'])] = {
            'title': row.get('title') or '',
            'text': row.get('text') or '',
        }

    qrels = defaultdict(dict)
    for row in qrels_ds:
        qrels[str(row['query-id'])][str(row['corpus-id'])] = int(row['score'])

    valid_qids = set(qrels.keys())
    queries = {}
    for row in queries_ds:
        qid = str(row['_id'])
        if qid in valid_qids:
            queries[qid] = {'text': row.get('text') or ''}
    qrels = {q: r for q, r in qrels.items() if q in queries}

    print(f'{name}: corpus={len(corpus)}, queries={len(queries)}, qrels={len(qrels)}')
    return corpus, queries, dict(qrels)


DATASETS = {
    'rus-scifact': load_beir_like('rus-scifact', 'test'),
    'rus-nfcorpus': load_beir_like('rus-nfcorpus', 'test'),
}

## 2. Препроцессинг для BM25

Для русского IR удалять стоп-слова надо аккуратно: убираем местоимения / союзы / частицы / предлоги, но **сохраняем** вопросительные слова (`что`, `как`, `какой`, `где`, `когда`, `почему`, `сколько` и т.д.) и отрицания (`не`, `нет`, `ни`) — они несут смысл в IR-запросах.

Список ниже — это nltk-russian минус вопросительные/отрицания/значимые наречия.

In [ ]:
# Список «безопасных» стоп-слов: без вопросительных слов и отрицаний.
# Базируется на nltk-russian, очищен от информативных токенов.
SOFT_RU_STOPWORDS = frozenset("""
и в во на с со от до из у к о об про над под при за по для
а но или либо да же ли бы б
я ты он она оно мы вы они меня тебя его её нас вас их
мне тебе ему ей нам вам им мной тобой нами вами ими
себя себе собой
это эта этот эти то та тот те
был была было были быть есть
только лишь уже ещё тоже также
вот ведь ну ах ой эх
если иначе чтобы хотя пока
""".split())

_morph = pymorphy3.MorphAnalyzer()
_TOKEN_RE = re.compile(r'[а-яёa-z0-9]+', re.IGNORECASE)
_lemma_cache: dict[str, str] = {}


def _lemma(token: str) -> str:
    cached = _lemma_cache.get(token)
    if cached is not None:
        return cached
    cached = _morph.parse(token)[0].normal_form
    _lemma_cache[token] = cached
    return cached


def preprocess(text: str) -> list[str]:
    """lowercase → tokenize → strip punct → lemmatize → soft stopword filter."""
    text = text.lower()
    out = []
    for raw in _TOKEN_RE.findall(text):
        if len(raw) < 2:
            continue
        lemma = _lemma(raw)
        if lemma in SOFT_RU_STOPWORDS:
            continue
        out.append(lemma)
    return out


def preprocess_str(text: str) -> str:
    return ' '.join(preprocess(text))


# Smoke test
print(preprocess_str('Какие методы машинного обучения используются для анализа геномных данных?'))

## 3. Метрики (pytrec_eval)

In [ ]:
K_VALUES = [10, 100]


def evaluate_run(qrels: dict, run: dict, k_values=K_VALUES):
    metric_strs = set()
    for k in k_values:
        metric_strs.add(f'ndcg_cut.{k}')
        metric_strs.add(f'recall.{k}')
        metric_strs.add(f'map_cut.{k}')
    evaluator = pytrec_eval.RelevanceEvaluator(qrels, metric_strs)
    per_q = evaluator.evaluate(run)
    agg = defaultdict(list)
    for q, m in per_q.items():
        for k, v in m.items():
            agg[k].append(v)
    means = {k: float(np.mean(v)) for k, v in agg.items()}
    out = {}
    for k in k_values:
        out[f'NDCG@{k}'] = means.get(f'ndcg_cut_{k}', 0.0)
        out[f'Recall@{k}'] = means.get(f'recall_{k}', 0.0)
        out[f'MAP@{k}'] = means.get(f'map_cut_{k}', 0.0)
    return out

## 4. Препроцессинг корпуса + BM25-ретривер

Корпус препроцессится тем же `preprocess()`, что и запросы. Делается это **один раз на датасет** и сохраняется в `cache/corpus_proc_{dataset}.json`. После этого BM25-индекс строится поверх этих токенов.

In [ ]:
_BM25_INDEX_CACHE: dict[str, tuple] = {}


def preprocess_corpus(corpus: dict, dataset_name: str) -> dict[str, list[str]]:
    """Прогоняет каждый документ через preprocess(title + ' ' + text).
    Кэшируется на диск — на nfcorpus 3.6k док-ов это ~10–30 секунд первый раз,
    далее моментально.
    Возвращает: {doc_id: [token, ...]}
    """
    cache = CACHE_DIR / f'corpus_proc_{dataset_name}.json'
    if cache.exists():
        return json.loads(cache.read_text(encoding='utf-8'))
    print(f'  [preprocess] tokenizing+lemmatizing corpus for {dataset_name} '
          f'({len(corpus):,} docs)...')
    out: dict[str, list[str]] = {}
    for did, doc in tqdm(corpus.items(), desc=f'preprocess/{dataset_name}'):
        text = (doc['title'] + ' ' + doc['text']).strip()
        out[did] = preprocess(text)
    cache.write_text(json.dumps(out, ensure_ascii=False), encoding='utf-8')
    return out


def get_bm25_index(corpus: dict, dataset_name: str):
    if dataset_name in _BM25_INDEX_CACHE:
        return _BM25_INDEX_CACHE[dataset_name]
    proc = preprocess_corpus(corpus, dataset_name)
    doc_ids = list(proc.keys())
    print(f'  [BM25] building index for {dataset_name} ({len(doc_ids):,} docs)...')
    bm25 = BM25Okapi([proc[d] for d in doc_ids])
    _BM25_INDEX_CACHE[dataset_name] = (bm25, doc_ids)
    return bm25, doc_ids


def bm25_search(corpus: dict, dataset_name: str,
                processed_queries: dict, k: int = 100) -> dict:
    """processed_queries: {qid: list[token]} — запросы УЖЕ предобработанные."""
    bm25, doc_ids = get_bm25_index(corpus, dataset_name)
    run = {}
    for qid, q_tokens in tqdm(processed_queries.items(), desc=f'BM25 search {dataset_name}'):
        if not q_tokens:
            run[qid] = {}
            continue
        scores = bm25.get_scores(q_tokens)
        kk = min(k, len(scores))
        top_idx = np.argpartition(-scores, kk - 1)[:kk]
        top_idx = top_idx[np.argsort(-scores[top_idx])]
        run[qid] = {doc_ids[i]: float(scores[i]) for i in top_idx}
    return run

## 5. Giga-embeddings ретривер

Грузим один раз, кэшируем эмбеддинги документов на диск, потом для каждого QE-метода энкодим только запросы. По условию задачи (RRF вариант B) giga получает **оригинальный** запрос — не расширенный.

In [ ]:
GIGA_HF_ID = 'ai-sage/Giga-Embeddings-instruct'
GIGA_QUERY_PREFIX = ('Instruct: Дан вопрос, найди подходящий документ '
                    'который отвечает на вопрос\nQuestion: ')
GIGA_BATCH = 16
GIGA_MAX_LEN = 4096
GIGA_DTYPE = torch.bfloat16

_GIGA_MODEL = None
_GIGA_TOK = None


def load_giga():
    global _GIGA_MODEL, _GIGA_TOK
    if _GIGA_MODEL is not None:
        return _GIGA_MODEL, _GIGA_TOK
    print(f'Loading {GIGA_HF_ID} ...')
    _GIGA_TOK = AutoTokenizer.from_pretrained(GIGA_HF_ID, trust_remote_code=True)
    kw = dict(trust_remote_code=True, torch_dtype=GIGA_DTYPE)
    try:
        _GIGA_MODEL = AutoModel.from_pretrained(
            GIGA_HF_ID, attn_implementation='flash_attention_2', **kw
        ).to(DEVICE)
    except (ImportError, ValueError) as e:
        print(f'  flash_attention_2 unavailable ({type(e).__name__}); using eager.')
        _GIGA_MODEL = AutoModel.from_pretrained(GIGA_HF_ID, **kw).to(DEVICE)
    _GIGA_MODEL.eval()
    return _GIGA_MODEL, _GIGA_TOK


@torch.inference_mode()
def giga_encode(texts: list[str], prefix: str = '', desc: str = 'giga') -> np.ndarray:
    model, tok = load_giga()
    if prefix:
        texts = [prefix + t for t in texts]
    order = np.argsort([len(t) for t in texts])
    sorted_texts = [texts[i] for i in order]
    out = [None] * len(texts)
    for s in tqdm(range(0, len(sorted_texts), GIGA_BATCH), desc=desc):
        batch = sorted_texts[s:s + GIGA_BATCH]
        enc = tok(batch, padding=True, truncation=True,
                  max_length=GIGA_MAX_LEN, return_tensors='pt').to(DEVICE)
        emb = model(**enc, return_embeddings=True)
        if not torch.is_tensor(emb):
            emb = emb[0] if isinstance(emb, (tuple, list)) else emb.embeddings
        emb = torch.nn.functional.normalize(emb, p=2, dim=1).float().cpu().numpy()
        for i, idx in enumerate(order[s:s + GIGA_BATCH]):
            out[idx] = emb[i]
    return np.vstack(out)


def faiss_search_gpu(doc_emb: np.ndarray, query_emb: np.ndarray, k: int = 100):
    index = faiss.IndexFlatIP(doc_emb.shape[1])
    res = faiss.StandardGpuResources()
    index = faiss.index_cpu_to_gpu(res, 0, index)
    index.add(doc_emb.astype(np.float32))
    return index.search(query_emb.astype(np.float32), k)


def get_giga_doc_emb(corpus: dict, dataset_name: str) -> tuple[np.ndarray, list[str]]:
    cache = CACHE_DIR / f'giga_doc_{dataset_name}.npy'
    cache_ids = CACHE_DIR / f'giga_doc_ids_{dataset_name}.json'
    if cache.exists() and cache_ids.exists():
        return np.load(cache), json.loads(cache_ids.read_text())
    doc_ids = list(corpus.keys())
    doc_texts = [(corpus[d]['title'] + ' ' + corpus[d]['text']).strip() for d in doc_ids]
    emb = giga_encode(doc_texts, prefix='', desc=f'giga/docs/{dataset_name}')
    np.save(cache, emb)
    cache_ids.write_text(json.dumps(doc_ids))
    return emb, doc_ids


def giga_search(corpus: dict, dataset_name: str,
                qid2text: dict, k: int = 100) -> dict:
    """qid2text: {qid: str} — сырые запросы (без препроцессинга, БЕЗ префикса)."""
    doc_emb, doc_ids = get_giga_doc_emb(corpus, dataset_name)
    qids = list(qid2text.keys())
    q_texts = [qid2text[q] for q in qids]
    q_emb = giga_encode(q_texts, prefix=GIGA_QUERY_PREFIX, desc=f'giga/queries/{dataset_name}')
    scores, idxs = faiss_search_gpu(doc_emb, q_emb, k=k)
    run = {}
    for i, qid in enumerate(qids):
        run[qid] = {}
        for j in range(idxs.shape[1]):
            di = idxs[i, j]
            if di < 0:
                continue
            run[qid][doc_ids[di]] = float(scores[i, j])
    return run

## 6. RRF (вариант B)

Сливаем `giga(оригинальный_запрос)` и `bm25(расширенный+предобработанный_запрос)`. Стандартный k=60.

In [ ]:
def rrf_fuse(runs: list[dict], k_rrf: int = 60, top_k: int = 100) -> dict:
    qids = set().union(*[set(r.keys()) for r in runs])
    fused = {}
    for qid in qids:
        agg = defaultdict(float)
        for r in runs:
            if qid not in r:
                continue
            ranked = sorted(r[qid].items(), key=lambda x: -x[1])
            for rank, (did, _) in enumerate(ranked, start=1):
                agg[did] += 1.0 / (k_rrf + rank)
        top = sorted(agg.items(), key=lambda x: -x[1])[:top_k]
        fused[qid] = {d: float(s) for d, s in top}
    return fused

## 7. DeepSeek API: drop-in замена для vLLM-обёрток

Все четыре `method_*` из `qe_methods.ipynb` написаны под vLLM. Дадим им совместимый интерфейс через DeepSeek API.

* `make_llm()` возвращает `(client, sp_cls)`, где `client` — наш wrapper, а `sp_cls` — фейковый класс с теми же атрибутами, что у `vllm.SamplingParams` (используется только для совместимости сигнатур).
* `vllm_generate(llm, sp_cls, prompts, ...)` и `vllm_sample_n_batch(...)` — async-под-капотом, конкурентность через семафор.

DeepSeek API совместимо с OpenAI SDK — используем `AsyncOpenAI` с `base_url`.

In [ ]:
from openai import AsyncOpenAI


class _SamplingParams:
    """Имитация vllm.SamplingParams — мы используем только поля, которые читаем сами."""
    def __init__(self, temperature=0.0, top_p=1.0, max_tokens=128, n=1):
        self.temperature = temperature
        self.top_p = top_p
        self.max_tokens = max_tokens
        self.n = n


class DeepSeekClient:
    def __init__(self, api_key: str, base_url: str = DEEPSEEK_BASE_URL,
                 model: str = DEEPSEEK_MODEL, max_concurrency: int = DEEPSEEK_MAX_CONCURRENCY):
        self.client = AsyncOpenAI(api_key=api_key, base_url=base_url)
        self.model = model
        self.sem = asyncio.Semaphore(max_concurrency)

    async def _one(self, messages, sp: _SamplingParams, n_override: int | None = None):
        # n>1 эмулируем серией параллельных запросов с одним и тем же promptом
        # (DeepSeek не всегда корректно поддерживает n>1).
        n = n_override if n_override is not None else sp.n
        if n == 1:
            async with self.sem:
                for attempt in range(4):
                    try:
                        resp = await self.client.chat.completions.create(
                            model=self.model,
                            messages=messages,
                            temperature=sp.temperature,
                            top_p=sp.top_p,
                            max_tokens=sp.max_tokens,
                        )
                        return [resp.choices[0].message.content or '']
                    except Exception as e:
                        if attempt == 3:
                            print(f'  [deepseek] error after retries: {e}')
                            return ['']
                        await asyncio.sleep(2 ** attempt)
        else:
            tasks = [self._one(messages, sp, n_override=1) for _ in range(n)]
            chunks = await asyncio.gather(*tasks)
            return [c[0] for c in chunks]

    async def _batch(self, all_messages, sp: _SamplingParams):
        """Возвращает list[list[str]] длиной len(all_messages); каждый внутренний список длины sp.n."""
        tasks = [self._one(m, sp) for m in all_messages]
        return await asyncio.gather(*tasks)

    def get_tokenizer(self):
        # Совместимость с vllm-style render_chat — возвращаем None, render_chat ниже
        # обработает это и просто отдаст messages в OpenAI формат.
        return None


def make_llm():
    if not DEEPSEEK_API_KEY_PLACEHOLDER        raise RuntimeError('Set DEEPSEEK_API_KEY env var before using QE methods.')
    return DeepSeekClient(DEEPSEEK_API_KEY), _SamplingParams

In [ ]:
# --- Drop-in замена render_chat / vllm_generate / vllm_sample_n_batch ---

def _to_messages(p):
    """Принимает либо list[dict] (chat), либо str (одно user-сообщение)."""
    if isinstance(p, list):
        return p
    return [{'role': 'user', 'content': p}]


def _run_async(coro):
    """Запускает asyncio coroutine из sync-кода (включая Jupyter)."""
    try:
        loop = asyncio.get_event_loop()
    except RuntimeError:
        loop = None
    if loop and loop.is_running():
        # Jupyter: используем nest_asyncio при необходимости
        import nest_asyncio
        nest_asyncio.apply()
        return loop.run_until_complete(coro)
    return asyncio.run(coro)


def render_chat(llm, messages_or_str):
    """Здесь это no-op — DeepSeek принимает messages в OpenAI формате напрямую."""
    return _to_messages(messages_or_str)


def vllm_generate(llm, sp_cls, prompts, max_new_tokens=180,
                  temperature=0.0, top_p=1.0):
    """1 sample на промпт. Возвращает list[str]."""
    if not prompts:
        return []
    sp = sp_cls(temperature=temperature, top_p=top_p, max_tokens=max_new_tokens, n=1)
    msgs = [_to_messages(p) for p in prompts]
    results = _run_async(llm._batch(msgs, sp))
    return [r[0].strip() for r in results]


def vllm_sample_n_batch(llm, sp_cls, prompts, n, max_new_tokens=64,
                        temperature=1.0, top_p=1.0):
    """n samples на промпт. Возвращает list[list[str]]."""
    if not prompts:
        return []
    sp = sp_cls(temperature=temperature, top_p=top_p, max_tokens=max_new_tokens, n=n)
    msgs = [_to_messages(p) for p in prompts]
    results = _run_async(llm._batch(msgs, sp))
    return [[s.strip() for s in r] for r in results]

## 8. QE-методы (адаптировано из qe_methods.ipynb)

Прямой портинг `method_q2d`, `method_prompt_prf`, `method_pqr`, `method_word2passage` — теперь работают через DeepSeek-обёртки выше.

In [ ]:
# === Query2doc ============================================================
Q2D_INSTRUCTION = 'Напиши абзац, отвечающий на заданный запрос.'
Q2D_FEWSHOT = [
    ('симптомы дефицита витамина D у взрослых',
     'Дефицит витамина D у взрослых проявляется болями в мышцах и костях, '
     'хронической усталостью, частыми простудами и снижением плотности костной ткани.'),
    ('что такое квантовая запутанность',
     'Квантовая запутанность — явление, при котором состояния двух или более частиц '
     'связаны так, что измерение одной мгновенно определяет состояние другой.'),
    ('как работает протокол TLS handshake',
     'TLS handshake устанавливает защищённое соединение: стороны согласуют шифры, '
     'обмениваются сертификатами, выводят сеансовый ключ через ECDHE.'),
    ('влияние ферментации на пищевую ценность сои',
     'Ферментация сои разрушает антинутриенты, повышает биодоступность белков, '
     'образует биоактивные пептиды и витамины группы B.'),
]


def _q2d_user(query: str) -> str:
    parts = [Q2D_INSTRUCTION, '']
    for q, p in Q2D_FEWSHOT:
        parts += [f'Запрос: {q}', f'Абзац: {p}', '']
    parts += [f'Запрос: {query}', 'Абзац:']
    return '\n'.join(parts)


def method_q2d(qtexts, *, llm, sp_cls,
               max_new_tokens=128, temperature=1.0):
    if not qtexts:
        return []
    print(f'  [Q2D] generating for {len(qtexts):,} queries ...')
    prompts = [_q2d_user(q) for q in qtexts]
    outs = vllm_generate(llm, sp_cls, prompts,
                         max_new_tokens=max_new_tokens, temperature=temperature)
    cleaned = []
    for text in outs:
        for stop in ('\nЗапрос:', '\nАбзац:'):
            if stop in text:
                text = text.split(stop)[0]
        cleaned.append(text.strip())
    return cleaned

In [ ]:
# === PromptPRF ============================================================
PRF_DEPTH = 5
PROMPTPRF_TEMPLATES = {
    'keywords': ('Текст: {p}\n\nНа основе текста составь маркированный список '
                 'релевантных ключевых слов:', 64, 'Ключевые слова'),
    'facts':    ('Текст: {p}\n\nНа основе текста составь маркированный список '
                 'релевантных фактов:', 256, 'Факты'),
    'entities': ('Текст: {p}\n\nНа основе текста составь маркированный список '
                 'релевантных именованных сущностей:', 64, 'Сущности'),
}


def method_prompt_prf(qtexts, *,
                      retriever_fn,
                      id2text,
                      llm, sp_cls,
                      feature_type='keywords',
                      prf_depth=PRF_DEPTH,
                      doc_truncate=1500):
    if not qtexts:
        return []
    template, mtok, fname = PROMPTPRF_TEMPLATES[feature_type]
    print(f'  [PromptPRF/{feature_type}] retrieving top-{prf_depth} ...')
    top_per_q = retriever_fn(qtexts, prf_depth)
    unique = sorted({d for v in top_per_q.values() for d in v})
    feat_cache = {}
    if unique:
        print(f'     extracting features for {len(unique):,} unique docs ...')
        prompts = [template.replace('{p}', id2text.get(d, '')[:doc_truncate]) for d in unique]
        outs = vllm_generate(llm, sp_cls, prompts, max_new_tokens=mtok, temperature=0.0)
        feat_cache = {d: t.strip() for d, t in zip(unique, outs)}
    out = []
    for qi in range(len(qtexts)):
        parts = []
        for rank, did in enumerate(top_per_q.get(qi, []), start=1):
            f = feat_cache.get(did, '')
            if f:
                parts.append(f'{fname} для топ-{rank} документа: {f}.')
        out.append(' '.join(parts))
    return out

In [ ]:
# === PQR ==================================================================
PQR_TEMP = 1.2
PQR_MAX_TOK = 28
PQR_N = 16
PQR_K_RANGE = (2, 3, 4, 5, 6)
PQR_ZS_INSTR = ('Перефразируй поисковый запрос так, чтобы он по смыслу совпадал '
                'с исходным. Выдавай только переформулировку.')
PQR_TOPIC_S1 = ('Перечисли через запятую 3-5 ключевых аспектов следующего '
                'поискового запроса. Без пояснений.')
PQR_TOPIC_S2 = ('Сформулируй короткий поисковый запрос об аспекте «{TOPIC}» '
                'в контексте: «{QUERY}». Выдавай только запрос.')


def _pqr_fit_gmm(emb, k_range=PQR_K_RANGE):
    from sklearn.mixture import GaussianMixture
    n = emb.shape[0]
    best, best_bic = None, float('inf')
    for K in k_range:
        if K >= n:
            break
        try:
            gm = GaussianMixture(n_components=K, covariance_type='diag',
                                 max_iter=100, n_init=2,
                                 random_state=0, reg_covar=1e-4)
            gm.fit(emb)
            b = gm.bic(emb)
            if b < best_bic:
                best_bic, best = b, gm
        except Exception:
            continue
    if best is None:
        gm = GaussianMixture(n_components=1, covariance_type='diag',
                             max_iter=100, random_state=0, reg_covar=1e-4)
        gm.fit(emb)
        return gm
    return best


def method_pqr(qtexts, *, llm, sp_cls, encode_fn):
    if not qtexts:
        return []
    print(f'  [PQR] sampling for {len(qtexts):,} queries ...')
    zs_prompts = [
        [{'role': 'system', 'content': PQR_ZS_INSTR},
         {'role': 'user',   'content': q}] for q in qtexts
    ]
    zs_per_q = vllm_sample_n_batch(llm, sp_cls, zs_prompts, n=PQR_N,
                                   max_new_tokens=PQR_MAX_TOK,
                                   temperature=PQR_TEMP, top_p=1.0)

    topic_prompts = [
        [{'role': 'system', 'content': PQR_TOPIC_S1},
         {'role': 'user',   'content': q}] for q in qtexts
    ]
    topics_raw = vllm_generate(llm, sp_cls, topic_prompts, max_new_tokens=64,
                               temperature=0.0)

    s2_prompts, s2_route = [], []
    for qi, (q, raw) in enumerate(zip(qtexts, topics_raw)):
        topics = [t.strip(' *•-—\t"\'.') for t in re.split(r'[,;\n]+', raw)]
        topics = [t for t in topics if 1 <= len(t.split()) <= 6][:5]
        for topic in topics:
            prompt = PQR_TOPIC_S2.replace('{TOPIC}', topic).replace('{QUERY}', q)
            s2_prompts.append([{'role': 'user', 'content': prompt}])
            s2_route.append(qi)
    s2_n = max(2, PQR_N // 5)
    s2_per_p = (vllm_sample_n_batch(llm, sp_cls, s2_prompts, n=s2_n,
                                    max_new_tokens=PQR_MAX_TOK,
                                    temperature=PQR_TEMP, top_p=1.0)
                if s2_prompts else [])
    ta_per_q = {qi: [] for qi in range(len(qtexts))}
    for qi, samples in zip(s2_route, s2_per_p):
        ta_per_q[qi].extend(s for s in samples if s)

    print('  [PQR] embedding samples + GMM ...')
    out = []
    for qi in tqdm(range(len(qtexts)), desc='  PQR-GMM'):
        all_samp = [s for s in (list(zs_per_q[qi]) + ta_per_q.get(qi, [])) if s]
        if len(all_samp) < 2:
            out.append(qtexts[qi])
            continue
        emb = encode_fn(all_samp).astype(np.float32)
        gm = _pqr_fit_gmm(emb)
        means = gm.means_.astype(np.float32)
        means /= np.linalg.norm(means, axis=1, keepdims=True) + 1e-9
        sim = emb @ means.T
        rep = sim.argmax(axis=0)
        out.append(' '.join(all_samp[i] for i in rep))
    return out

In [ ]:
# === Word2Passage =========================================================
W2P_QUERY_TYPES = ('description', 'person', 'entity', 'numeric', 'location')
W2P_SIGNIFICANCE = {
    'description': (0.72, 0.57, 0.97), 'entity': (0.50, 0.73, 0.48),
    'person': (1.10, 1.08, 0.70), 'numeric': (0.78, 1.15, 0.83),
    'location': (1.00, 1.00, 0.73),
}
W2P_DEFAULT_SIG = (1.0, 1.0, 1.0)
W2P_GEN_PROMPT = (
    'Сгенерируй абзац, предложение и список слов, отвечающих на ЗАПРОС. '
    'Термины, важные для ответа, должны часто встречаться во всех трёх частях.\n'
    '### Определения:\n'
    '**passage**: ответь информативным связным абзацем.\n'
    '**sentence**: ответь одним содержательным предложением.\n'
    '**word**: ответь списком слов.\n'
    '### ЗАПРОС:\n{QUERY}\n'
    '### СТРОГИЙ ФОРМАТ (только JSON):\n'
    '{"passage": "ваш абзац", "sentence": "ваше предложение", "word": ["слово1", "слово2"]}'
)
W2P_TYPE_PROMPT = (
    'Дан набор поисковых запросов разных типов:\n'
    'Тип: description — Запрос: причины воспаления тазовой области\n'
    'Тип: numeric — Запрос: средняя зарплата консультанта по семейным делам\n'
    'Тип: location — Запрос: какой самый большой континент\n'
    'Тип: entity — Запрос: какие растения растут в Орегоне\n'
    'Тип: person — Запрос: актёрский состав фильма «Интерстеллар»\n\n'
    'Классифицируй следующий запрос: [description, numeric, location, entity, person].\n'
    'Запрос: {QUERY}\n\n'
    '### ФОРМАТ ОТВЕТА\nТип запроса: ваш ответ'
)
_W2P_TOK = re.compile(r'[\w]+', re.UNICODE)


def _w2p_tok(t):
    return [s.lower() for s in _W2P_TOK.findall(t or '')]


def _w2p_parse_json(raw):
    s = (raw or '').strip()
    s = re.sub(r'^```(?:json)?\s*', '', s)
    s = re.sub(r'\s*```$', '', s)
    m = re.search(r'\{.*\}', s, re.DOTALL)
    if m:
        s = m.group(0)
    try:
        d = json.loads(s)
    except Exception:
        d = {}
    return {
        'passage': str(d.get('passage', '') or ''),
        'sentence': str(d.get('sentence', '') or ''),
        'word': [str(w) for w in (d.get('word', []) or []) if w],
    }


def _w2p_avg_unique(corpus_texts, sample_n=2000):
    txts = corpus_texts[:sample_n]
    return float(np.mean([len(set(_w2p_tok(t))) for t in txts])) if txts else 1.0


def method_word2passage(qtexts, *, corpus_texts, llm, sp_cls,
                        n_refs=3, alpha=1.0, repeat_scale=5):
    if not qtexts:
        return []
    print(f'  [W2P] generating refs for {len(qtexts):,} queries ...')
    W = _w2p_avg_unique(corpus_texts)

    prompts = [
        [{'role': 'user', 'content': W2P_GEN_PROMPT.replace('{QUERY}', q)}]
        for q in qtexts
    ]
    refs_raw = vllm_sample_n_batch(llm, sp_cls, prompts, n=n_refs,
                                   max_new_tokens=512, temperature=0.7, top_p=0.9)
    type_prompts = [
        [{'role': 'user', 'content': W2P_TYPE_PROMPT.replace('{QUERY}', q)}]
        for q in qtexts
    ]
    type_raw = vllm_generate(llm, sp_cls, type_prompts, max_new_tokens=20, temperature=0.0)

    out = []
    for q, raw_refs, raw_type in zip(qtexts, refs_raw, type_raw):
        refs = [_w2p_parse_json(r) for r in raw_refs]
        s_lower = (raw_type or '').lower()
        q_type = next((c for c in W2P_QUERY_TYPES if c in s_lower), 'description')
        I_qw, I_qs, I_qp = W2P_SIGNIFICANCE.get(q_type, W2P_DEFAULT_SIG)
        word_R, word_R_freq = {}, {}
        for r in refs:
            cw = Counter(_w2p_tok(' '.join(r['word'])))
            cs = Counter(_w2p_tok(r['sentence']))
            cp = Counter(_w2p_tok(r['passage']))
            for t in set(cw) | set(cs) | set(cp):
                score = I_qw * cw.get(t, 0) + I_qs * cs.get(t, 0) + I_qp * cp.get(t, 0)
                word_R[t] = word_R.get(t, 0.0) + score
                word_R_freq[t] = word_R_freq.get(t, 0) + cw[t] + cs[t] + cp[t]
        if W > 0:
            scale_R = alpha / math.sqrt(W)
            word_R = {t: s * scale_R for t, s in word_R.items()}
        q_freq = Counter(_w2p_tok(q))
        sumF_R = sum(word_R_freq.values())
        sumF_Q = sum(q_freq.values())
        if sumF_Q == 0:
            word_Q = {}
        else:
            norm = math.sqrt(sumF_R) / math.sqrt(sumF_Q) if sumF_R > 0 else 1.0
            word_Q = {t: norm * q_freq[t] for t in q_freq}
        weights = dict(word_R)
        for t, v in word_Q.items():
            weights[t] = weights.get(t, 0.0) + v
        if not weights:
            out.append('')
        else:
            items = sorted(weights.items(), key=lambda kv: -kv[1])[:200]
            max_I = items[0][1] or 1.0
            parts = []
            for t, I in items:
                n_rep = max(1, int(round((I / max_I) * repeat_scale)))
                parts.extend([t] * n_rep)
            out.append(' '.join(parts))
    return out

## 9. Генерация расширений

Расширения для каждого `(qe_method, dataset)` генерируем **один раз**, кэшируем как `expansions/__raw_{method}_{dataset}.json` (тройка `original → expanded`). Затем делаем 24 файла (по комбинациям) — они физически идентичны для bm25 и rrf, но это сделано для прозрачности артефактов.

Для **giga** в варианте RRF-B расширение по-прежнему генерируется (для вывода в файл и для bm25-ветки), но в search-функции giga получит **оригинальный** запрос. Для трека «giga + QE» (отдельный ретривер из 12 комбинаций) giga получает расширенный запрос как обычно.

In [ ]:
QE_METHODS = ['query2doc', 'promptprf', 'pqr', 'word2passage']
RETRIEVERS = ['bm25', 'giga', 'rrf']  # rrf == RRF(giga + bm25), вариант B


def make_prf_retriever(corpus, dataset_name, qid_list):
    """Для PromptPRF — нужен retriever_fn(qtexts, k) -> {qi: [doc_id]}.
    Используем BM25 (быстро, не требует LLM)."""
    bm25, doc_ids = get_bm25_index(corpus, dataset_name)

    def fn(qtexts, k):
        out = {}
        for qi, q in enumerate(qtexts):
            tokens = preprocess(q)
            if not tokens:
                out[qi] = []
                continue
            scores = bm25.get_scores(tokens)
            kk = min(k, len(scores))
            top_idx = np.argpartition(-scores, kk - 1)[:kk]
            top_idx = top_idx[np.argsort(-scores[top_idx])]
            out[qi] = [doc_ids[i] for i in top_idx]
        return out
    return fn


def make_pqr_encoder():
    """Для PQR — нужен encode_fn(list[str]) -> np.ndarray.
    Используем giga-embeddings (ту же, что в основном dense-ретривере)."""
    def fn(texts):
        return giga_encode(texts, prefix=GIGA_QUERY_PREFIX, desc='pqr/encode')
    return fn


def generate_expansions_for(method: str, corpus: dict, queries: dict,
                            dataset_name: str, llm, sp_cls) -> dict:
    """Возвращает {qid: expanded_text}. Кэшируется per (method, dataset)."""
    cache = EXP_DIR / f'__raw_{method}_{dataset_name}.json'
    if cache.exists():
        return json.loads(cache.read_text(encoding='utf-8'))

    qids = list(queries.keys())
    qtexts = [queries[q]['text'] for q in qids]

    print(f'\n=== {method} on {dataset_name} ({len(qids)} queries) ===')
    if method == 'query2doc':
        expansions = method_q2d(qtexts, llm=llm, sp_cls=sp_cls)
    elif method == 'promptprf':
        retriever_fn = make_prf_retriever(corpus, dataset_name, qids)
        id2text = {d: (corpus[d]['title'] + ' ' + corpus[d]['text']).strip()
                   for d in corpus}
        expansions = method_prompt_prf(
            qtexts, retriever_fn=retriever_fn, id2text=id2text,
            llm=llm, sp_cls=sp_cls, feature_type='keywords', prf_depth=PRF_DEPTH,
        )
    elif method == 'pqr':
        expansions = method_pqr(qtexts, llm=llm, sp_cls=sp_cls,
                                encode_fn=make_pqr_encoder())
    elif method == 'word2passage':
        corpus_texts = [(corpus[d]['title'] + ' ' + corpus[d]['text']).strip()
                        for d in corpus]
        expansions = method_word2passage(qtexts, corpus_texts=corpus_texts,
                                         llm=llm, sp_cls=sp_cls)
    else:
        raise ValueError(method)

    out = dict(zip(qids, expansions))
    cache.write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding='utf-8')
    return out

In [ ]:
# Генерируем 4 × 2 = 8 наборов расширений
# (это шаг с DeepSeek-вызовами — самый дорогой по времени и API).

llm, sp_cls = make_llm()

raw_expansions = {}  # {(method, dataset): {qid: expanded}}
for ds_name, (corpus, queries, _) in DATASETS.items():
    for method in QE_METHODS:
        t0 = time.time()
        raw_expansions[(method, ds_name)] = generate_expansions_for(
            method, corpus, queries, ds_name, llm, sp_cls
        )
        print(f'  done in {time.time() - t0:.1f}s')

## 10. Сохранение 24 файлов расширений

Для каждой комбинации `(retriever, qe_method, dataset)` сохраняем тройки `{qid, original, expanded, processed}`.

* Для **bm25** и для **giga** (как самостоятельного ретривера) поле `expanded` = `original + ' ' + expansion`, а `processed` = preprocess(expanded).
* Для **rrf** (вариант B): `expanded` = такой же как у bm25 (т.к. расширение применяется к bm25-ветке), `processed` тоже как у bm25; giga в RRF-B получает оригинальный запрос. В файл записываем то, что реально пойдёт в bm25-ветку RRF.

Итого 3 × 4 × 2 = **24 файла**.

In [ ]:
def build_expansion_records(retriever: str, method: str, dataset_name: str,
                            queries: dict, raw_exp: dict) -> list[dict]:
    """Строит список троек (qid, original, expanded, processed).

    `expanded` — это то, что пойдёт в ретривер ДО предобработки.
    Для bm25 и giga: original + ' ' + expansion.
    Для rrf (вариант B): то же самое — это то, что увидит bm25-ветка;
        giga-ветка получит original независимо.
    `processed` — preprocess(expanded).
    """
    records = []
    for qid in queries:
        original = queries[qid]['text']
        exp_text = raw_exp.get(qid, '')
        expanded = (original + ' ' + exp_text).strip() if exp_text else original
        processed = preprocess_str(expanded)
        records.append({
            'qid': qid,
            'original': original,
            'expanded': expanded,
            'processed': processed,
        })
    return records


def expansion_path(retriever: str, method: str, dataset_name: str) -> Path:
    return EXP_DIR / f'{retriever}_{method}_{dataset_name}.json'


def run_path(retriever: str, method: str, dataset_name: str) -> Path:
    return RUN_DIR / f'{retriever}_{method}_{dataset_name}.json'


for ds_name, (corpus, queries, _) in DATASETS.items():
    for method in QE_METHODS:
        raw = raw_expansions[(method, ds_name)]
        for retriever in RETRIEVERS:
            recs = build_expansion_records(retriever, method, ds_name, queries, raw)
            path = expansion_path(retriever, method, ds_name)
            path.write_text(json.dumps(recs, ensure_ascii=False, indent=2),
                            encoding='utf-8')

print('Saved expansion files:', len(list(EXP_DIR.glob('[!_]*.json'))))

## 11. Прогон 12 комбинаций × 2 датасета = 24 поиска

Сохраняем top-100 для каждого запроса в `runs/{retriever}_{method}_{dataset}.json`.

In [ ]:
all_metrics = []  # сводная таблица

for ds_name, (corpus, queries, qrels) in DATASETS.items():
    print(f'\n{"=" * 70}\nDataset: {ds_name}\n{"=" * 70}')

    # Кэшируем раз: эмбеддинги док-ов для giga + BM25-индекс
    get_bm25_index(corpus, ds_name)
    _ = get_giga_doc_emb(corpus, ds_name)

    # Заранее строим qid->original для giga (чтобы повторно не считать в RRF-B)
    qid2orig = {q: queries[q]['text'] for q in queries}
    giga_orig_run_path = CACHE_DIR / f'giga_orig_run_{ds_name}.json'
    if giga_orig_run_path.exists():
        giga_orig_run = json.loads(giga_orig_run_path.read_text())
    else:
        giga_orig_run = giga_search(corpus, ds_name, qid2orig, k=100)
        giga_orig_run_path.write_text(json.dumps(giga_orig_run))

    for method in QE_METHODS:
        # Загружаем сохранённые расширения (общие для всех 3 ретриверов)
        recs = json.loads(
            expansion_path('bm25', method, ds_name).read_text(encoding='utf-8')
        )
        # qid -> processed tokens (для bm25)
        qid2proc_tok = {r['qid']: r['processed'].split() for r in recs}
        # qid -> expanded raw (для giga в самостоятельном треке)
        qid2expanded = {r['qid']: r['expanded'] for r in recs}

        # ===== Ретривер: BM25 (на расширенном+предобработанном запросе) =====
        bm25_run = bm25_search(corpus, ds_name, qid2proc_tok, k=100)
        run_path('bm25', method, ds_name).write_text(
            json.dumps(bm25_run, ensure_ascii=False))
        m = evaluate_run(qrels, bm25_run)
        all_metrics.append({
            'dataset': ds_name, 'retriever': 'bm25', 'qe': method, **m
        })
        print(f'  bm25 + {method:13s}  NDCG@10={m["NDCG@10"]:.4f}  '
              f'R@100={m["Recall@100"]:.4f}')

        # ===== Ретривер: giga (на расширенном запросе как сыром тексте) =====
        giga_run = giga_search(corpus, ds_name, qid2expanded, k=100)
        run_path('giga', method, ds_name).write_text(
            json.dumps(giga_run, ensure_ascii=False))
        m = evaluate_run(qrels, giga_run)
        all_metrics.append({
            'dataset': ds_name, 'retriever': 'giga', 'qe': method, **m
        })
        print(f'  giga + {method:13s}  NDCG@10={m["NDCG@10"]:.4f}  '
              f'R@100={m["Recall@100"]:.4f}')

        # ===== Ретривер: RRF(giga[orig] + bm25[expanded]) =====
        rrf_run = rrf_fuse([giga_orig_run, bm25_run], k_rrf=60, top_k=100)
        run_path('rrf', method, ds_name).write_text(
            json.dumps(rrf_run, ensure_ascii=False))
        m = evaluate_run(qrels, rrf_run)
        all_metrics.append({
            'dataset': ds_name, 'retriever': 'rrf', 'qe': method, **m
        })
        print(f'  rrf  + {method:13s}  NDCG@10={m["NDCG@10"]:.4f}  '
              f'R@100={m["Recall@100"]:.4f}')

    # Освобождаем GPU между датасетами (на случай большого корпуса nfcorpus)
    gc.collect()
    torch.cuda.empty_cache()

print(f'\nSaved run files: {len(list(RUN_DIR.glob("*.json")))}')

## 12. Сводная таблица

In [ ]:
import pandas as pd

df = pd.DataFrame(all_metrics)
metric_cols = ['NDCG@10', 'NDCG@100', 'Recall@10', 'Recall@100', 'MAP@10', 'MAP@100']
for col in metric_cols:
    df[col] = df[col].round(4)
df = df[['dataset', 'retriever', 'qe'] + metric_cols]
df.sort_values(['dataset', 'retriever', 'qe'], inplace=True)
df.to_csv(RES_DIR / 'qe_summary.csv', index=False)
df

In [ ]:
# Удобный wide-формат: ретриверы как строки, qe-методы как колонки, метрика — NDCG@10
wide = df.pivot_table(
    index=['dataset', 'retriever'], columns='qe', values='NDCG@10'
).round(4)
wide